# Fake Job Posting Detector

NLP + metadata classifier to detect fraudulent job postings.

**Dataset:** Kaggle "Real or Fake Job Postings" by Shivam Bansal  
**Target:** `fraudulent` (0 = real, 1 = fake) — 866 fake (4.84%) vs 17,014 real

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    recall_score,
)
from scipy.sparse import hstack
import scipy.sparse as sp

sns.set_style("whitegrid")
np.random.seed(42)

## Phase 1 — Load & Clean Data

In [ ]:
# Load the dataset
df = pd.read_csv('fake_job_postings.csv')
print(f"Dataset shape: {df.shape}")
print(f"Fraudulent postings: {df['fraudulent'].sum()} / {len(df)} "
      f"({df['fraudulent'].mean() * 100:.2f}%)\n")

# Combine the 5 text columns into one corpus
text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
df[text_cols] = df[text_cols].fillna('')
df['combined_text'] = df[text_cols].apply(
    lambda row: ' '.join(row.values), axis=1
)

# Keep only the columns we need for modeling
features_df = df[
    ['combined_text', 'telecommuting', 'has_company_logo', 'has_questions', 'fraudulent']
].copy()

print(f"Features shape: {features_df.shape}")
print(f"Fraud distribution:\n{features_df['fraudulent'].value_counts()}")

## Phase 2 — Feature Engineering

In [ ]:
# TF-IDF on combined text — captures important words and phrases
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),  # unigrams + bigrams
)
X_text = tfidf.fit_transform(features_df['combined_text'])
print(f"TF-IDF matrix shape: {X_text.shape}")

# Binary metadata features — convert to sparse format for stacking
X_meta = sp.csr_matrix(
    features_df[['telecommuting', 'has_company_logo', 'has_questions']].values
)

# Stack text features + metadata into a single feature matrix
X = hstack([X_text, X_meta])
y = features_df['fraudulent'].values

print(f"Full feature matrix shape: {X.shape}")
print(f"Total features: {X.shape[1]} (5000 text + 3 metadata)")

## Phase 3 — Modeling

In [ ]:
# Train/test split with stratification to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Train fraud rate: {y_train.mean() * 100:.2f}%")
print(f"Test fraud rate:  {y_test.mean() * 100:.2f}%\n")

# ── Model 1: Logistic Regression ─────────────────────────────────────────────
print("=" * 40)
print("Logistic Regression")
print("=" * 40)

lr = LogisticRegression(
    class_weight='balanced',  # handles imbalance automatically
    max_iter=1000,
    random_state=42,
)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

lr_roc_auc = roc_auc_score(y_test, y_prob_lr)
print(classification_report(y_test, y_pred_lr, target_names=['Real', 'Fake']))
print(f"ROC-AUC: {lr_roc_auc:.4f}\n")

# ── Model 2: Random Forest ───────────────────────────────────────────────────
print("=" * 40)
print("Random Forest")
print("=" * 40)

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

rf_roc_auc = roc_auc_score(y_test, y_prob_rf)
print(classification_report(y_test, y_pred_rf, target_names=['Real', 'Fake']))
print(f"ROC-AUC: {rf_roc_auc:.4f}\n")

# ── Pick the best model based on Fake-class Recall ───────────────────────────
lr_recall_fake = recall_score(y_test, y_pred_lr, pos_label=1)
rf_recall_fake = recall_score(y_test, y_pred_rf, pos_label=1)

if rf_recall_fake >= lr_recall_fake:
    best_model, best_name = rf, 'Random Forest'
    best_pred, best_prob = y_pred_rf, y_prob_rf
    best_recall = rf_recall_fake
else:
    best_model, best_name = lr, 'Logistic Regression'
    best_pred, best_prob = y_pred_lr, y_prob_lr
    best_recall = lr_recall_fake

print(f"★ Best model: {best_name} (Fake Recall = {best_recall:.4f})")

## Phase 4 — Evaluation Plots

In [ ]:
# ── Plot 1: Confusion Matrix (best model) ────────────────────────────────────
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Real', 'Fake'],
    yticklabels=['Real', 'Fake'],
)
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: ROC Curve (both models) ──────────────────────────────────────────
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_roc_auc:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Top 20 Words Predicting Fake Postings (from LR coefficients) ─────
feature_names = tfidf.get_feature_names_out().tolist() + [
    'telecommuting', 'has_company_logo', 'has_questions'
]
coefs = lr.coef_[0]

top20_idx = np.argsort(coefs)[-20:][::-1]
top20_words = [feature_names[i] for i in top20_idx]
top20_scores = [coefs[i] for i in top20_idx]

plt.figure(figsize=(8, 6))
sns.barplot(x=top20_scores, y=top20_words, palette='Reds_r', hue=top20_words, legend=False)
plt.title('Top 20 Words Predicting Fake Job Postings')
plt.xlabel('Logistic Regression Coefficient')
plt.tight_layout()
plt.savefig('top_words.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 4: Class Distribution ───────────────────────────────────────────────
plt.figure(figsize=(5, 4))
sns.countplot(x=features_df['fraudulent'], palette=['steelblue', 'crimson'], hue=features_df['fraudulent'], legend=False)
plt.xticks([0, 1], ['Real (17,014)', 'Fake (866)'])
plt.title('Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

## Phase 5 — Save Models

In [ ]:
joblib.dump(lr, 'fake_job_detector_lr.pkl')
joblib.dump(rf, 'fake_job_detector_rf.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("Models saved successfully")

## Phase 6 — Final Verification

In [ ]:
# Reload and verify models
lr_loaded = joblib.load('fake_job_detector_lr.pkl')
rf_loaded = joblib.load('fake_job_detector_rf.pkl')
tfidf_loaded = joblib.load('tfidf_vectorizer.pkl')
print("✓ Models loaded successfully from disk")

# Verify predictions match after reload
y_pred_lr_loaded = lr_loaded.predict(X_test)
y_pred_rf_loaded = rf_loaded.predict(X_test)
assert np.array_equal(y_pred_lr, y_pred_lr_loaded), "LR predictions don't match!"
assert np.array_equal(y_pred_rf, y_pred_rf_loaded), "RF predictions don't match!"
print("✓ Predictions identical after reload\n")

# Performance summary
print("=" * 60)
print("PERFORMANCE SUMMARY")
print("=" * 60)
print(f"{'Model':<25} {'Fake Recall':<15} {'Fake Prec':<15} {'Fake F1':<15} {'ROC-AUC':<10}")
print("-" * 80)

lr_report = classification_report(y_test, y_pred_lr, target_names=['Real', 'Fake'], output_dict=True)
rf_report = classification_report(y_test, y_pred_rf, target_names=['Real', 'Fake'], output_dict=True)

print(f"{'Logistic Regression':<25} {lr_report['Fake']['recall']:<15.4f} "
      f"{lr_report['Fake']['precision']:<15.4f} {lr_report['Fake']['f1-score']:<15.4f} "
      f"{lr_roc_auc:<10.4f}")
print(f"{'Random Forest':<25} {rf_report['Fake']['recall']:<15.4f} "
      f"{rf_report['Fake']['precision']:<15.4f} {rf_report['Fake']['f1-score']:<15.4f} "
      f"{rf_roc_auc:<10.4f}")

In [ ]:
# ── Sanity check with extreme examples ───────────────────────────────────────
print("Sanity checks on extreme examples")
print("-" * 40)

# Fake posting: urgency, vague language, no company details
fake_text = (
    "urgent hiring immediate salary paid work from home no experience "
    "need money fast earn thousands weekly guaranteed"
)
fake_vec = hstack([
    tfidf_loaded.transform([fake_text]),
    sp.csr_matrix([[1, 0, 0]]),  # telecommuting=1, logo=0, questions=0
])
fake_prob = lr_loaded.predict_proba(fake_vec)[0, 1]
print(f"Fake posting → predicts: {'FAKE' if fake_prob > 0.5 else 'REAL'} "
      f"(fraud probability: {fake_prob:.4f})")

# Real posting: specific skills, company details
real_text = (
    "software engineer required 5 years experience python java cloud "
    "competitive salary benefits health insurance 401k matching"
)
real_vec = hstack([
    tfidf_loaded.transform([real_text]),
    sp.csr_matrix([[0, 1, 1]]),  # telecommuting=0, logo=1, questions=1
])
real_prob = lr_loaded.predict_proba(real_vec)[0, 1]
print(f"Real posting → predicts: {'FAKE' if real_prob > 0.5 else 'REAL'} "
      f"(fraud probability: {real_prob:.4f})")

In [ ]:
# ── Success criteria ────────────────────────────────────────────────────────
print("=" * 40)
print("SUCCESS CRITERIA CHECK")
print("=" * 40)
print(f"Fake Recall ≥ 0.75 ? {'✅' if best_recall >= 0.75 else '❌'} ({best_recall:.4f})")
print(f"ROC-AUC    ≥ 0.95 ? {'✅' if lr_roc_auc >= 0.95 else '❌'} ({lr_roc_auc:.4f})")
print("=" * 40)
print()
print("Generated files:")
for f in ['confusion_matrix.png', 'roc_curve.png', 'top_words.png',
          'class_distribution.png', 'fake_job_detector_lr.pkl',
          'fake_job_detector_rf.pkl', 'tfidf_vectorizer.pkl']:
    if os.path.exists(f):
        print(f"  ✓ {f}")